# 03 – Holt-Winters (Leitfaden Punkt 8)

Ziel: Holt-Winters auf den Trainingsdaten anpassen (Parameter per Software), die 24 Testmonate prognostizieren und mit der Seasonal-Naive-Baseline unter denselben Bedingungen vergleichen (gleicher Split, gleiche Metriken aus `src/evaluation.py`).

Festgelegt vor der Modellierung:
- **Additive Saison, Periode 12**: aus `01_eda.ipynb` (Lag-12-Korrelation r = 0,89; Korrelation Amplitude–Niveau r = −0,23 auf den Trainingsdaten).
- **Trend**: additiv, ungedämpft oder gedämpft – Auswahl per AIC **nur auf den Trainingsdaten** (Abschnitt 2).
- Die Testdaten werden erst ab Abschnitt 4 verwendet.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
from src.data import load_umsatz
from src.evaluation import train_test_split, evaluate, seasonal_naive
from src import holtwinters as hw
from src.plots import speichern

s = load_umsatz(str(ROOT / "data/raw/45212-0004_de.csv"))
train, test = train_test_split(s)
print(f"Training: {train.index[0]:%m/%Y}–{train.index[-1]:%m/%Y} ({len(train)} Monate)")
print(f"Test:     {test.index[0]:%m/%Y}–{test.index[-1]:%m/%Y} ({len(test)} Monate)")

## 1. Modellgleichungen

Additives Holt-Winters-Verfahren in Komponentenform (Hyndman & Athanasopoulos 2021, wie in `statsmodels`). Es entspricht dem Verfahren aus Vorlesung 2; dort heißt das Niveau $\hat y^H$. Neu ist nur der Dämpfungsfaktor $\phi$ (ungedämpft: $\phi = 1$).

$$\ell_t = \alpha\,(y_t - s_{t-12}) + (1-\alpha)\,(\ell_{t-1} + \phi\, b_{t-1})$$
$$b_t = \beta\,(\ell_t - \ell_{t-1}) + (1-\beta)\,\phi\, b_{t-1}$$
$$s_t = \gamma\,(y_t - \ell_{t-1} - \phi\, b_{t-1}) + (1-\gamma)\,s_{t-12}$$

Prognose $h$ Monate nach dem Trainingsende $T$:

$$\hat y_{T+h|T} = \ell_T + (\phi + \phi^2 + \dots + \phi^h)\, b_T + s_{T+h-12k}$$

mit $s_{T+h-12k}$ = letzter geschätzter Saisonwert desselben Kalendermonats.

- $\ell_t$: Niveau, $b_t$: Trend (Steigung pro Monat), $s_t$: Saisonkomponente
- $\alpha, \beta, \gamma \in [0, 1]$: Glättungsparameter für Niveau, Trend, Saison
- $\phi \in (0, 1]$: Dämpfung; mit $\phi < 1$ flacht der Trend in der Prognose ab

`statsmodels` schätzt die Parameter und die Startwerte durch Minimierung der quadrierten Ein-Schritt-Fehler auf den Trainingsdaten. Dabei gilt die Nebenbedingung $\beta \le \alpha$.

## 2. Variantenwahl auf den Trainingsdaten (AIC)

Der AIC bewertet die Anpassung an die Trainingsdaten und bestraft zusätzliche Parameter. Kleiner ist besser. Die Wahl fällt hier, **bevor** die Testdaten benutzt werden.

In [ ]:
vergleich = hw.varianten_vergleichen(train)
auswahl = vergleich["AIC"].idxmin()
print("Auswahl per AIC:", auswahl)
vergleich.round(4)

**Interpretation:** Der gedämpfte Trend hat den kleineren AIC (668,34 gegenüber 678,26; AICc 670,52 gegenüber 680,22) und die kleinere Fehlerquadratsumme auf den Trainingsdaten (2062,8 gegenüber 2130,7), obwohl er einen Parameter mehr hat. → **Holt-Winters mit gedämpftem Trend** wird verwendet. Die Entscheidung beruht nur auf den Trainingsdaten.

## 3. Geschätzte Parameter des gewählten Modells

In [ ]:
res = hw.anpassen(train, gedaempft=(auswahl == "Holt-Winters (gedämpft)"))
par = hw.parameter(res)
print(par.round(4).to_string())
print()
print(f"beta = alpha (Randlösung der Nebenbedingung beta <= alpha): {abs(par['beta'] - par['alpha']) < 1e-3}")
print(f"Niveau am Trainingsende (08/2024): {res.level.iloc[-1]:.2f}")
print(f"Trend  am Trainingsende (08/2024): {res.trend.iloc[-1]:.3f} pro Monat")
print(f"Maximaler Trendbeitrag in der Prognose (phi/(1-phi) · b_T): {par['phi'] / (1 - par['phi']) * res.trend.iloc[-1]:.2f} Punkte")

**Interpretation:**

| Parameter | Wert | Bedeutung |
|---|---|---|
| $\alpha$ | 0,1648 | Das Niveau übernimmt 16,5 % der neuen, saisonbereinigten Beobachtung, der Rest kommt aus der Vorperiode → langsame Anpassung |
| $\beta$ | 0,1648 | Der Trend übernimmt 16,5 % der neuen Niveauänderung |
| $\gamma$ | 0,0893 | Die Saisonwerte übernehmen 8,9 % der neuen Abweichung → die Saisonfigur ändert sich nur langsam |
| $\phi$ | 0,9027 | Der Trend wird pro Prognosemonat mit 0,90 multipliziert |

- Am Trainingsende (08/2024): Niveau 93,94, Trend 0,183 Punkte pro Monat. Durch die Dämpfung trägt der Trend in der Prognose insgesamt höchstens 1,70 Punkte bei.
- $\beta$ ist gleich $\alpha$: Die Schätzung liegt am Rand der Nebenbedingung $\beta \le \alpha$ von `statsmodels`.

Modellgleichung mit den geschätzten Parametern:

$$\ell_t = 0{,}1648\,(y_t - s_{t-12}) + 0{,}8352\,(\ell_{t-1} + 0{,}9027\, b_{t-1})$$
$$b_t = 0{,}1648\,(\ell_t - \ell_{t-1}) + 0{,}8352 \cdot 0{,}9027\, b_{t-1}$$
$$s_t = 0{,}0893\,(y_t - \ell_{t-1} - 0{,}9027\, b_{t-1}) + 0{,}9107\, s_{t-12}$$

## 4. Prognose der 24 Testmonate

Ab hier werden die Testdaten verwendet. Das Modell wird nicht mehr verändert.

In [ ]:
p_hw = hw.prognose(res, len(test), "Holt-Winters (gedämpft)")
p_sn = seasonal_naive(train)

fig = hw.plot_prognose(train, test, {"Holt-Winters (gedämpft)": p_hw, "Seasonal Naive": p_sn})
speichern(fig, "06_hw_prognose")

In [ ]:
# Leistungskriterium: RMSE (Hauptmetrik), MAE, MAPE – gleiche Funktion für alle Modelle
res_ungedaempft = hw.anpassen(train, gedaempft=False)
p_hw0 = hw.prognose(res_ungedaempft, len(test), "Holt-Winters (ungedämpft)")

metriken = pd.DataFrame({
    "Holt-Winters (gedämpft)": evaluate(test, p_hw),
    "Seasonal Naive (Baseline)": evaluate(test, p_sn),
    "Holt-Winters (ungedämpft), nur zur Info": evaluate(test, p_hw0),
}).T
print(f"Trend am Trainingsende: gedämpft {res.trend.iloc[-1]:.3f}, ungedämpft {res_ungedaempft.trend.iloc[-1]:.3f} pro Monat")
print(f"Mittlerer Fehler (ŷ − y): gedämpft {(p_hw - test).mean():.2f}, ungedämpft {(p_hw0 - test).mean():.2f}, Seasonal Naive {(p_sn - test).mean():.2f}")
verbesserung = 1 - metriken.loc["Holt-Winters (gedämpft)", "RMSE"] / metriken.loc["Seasonal Naive (Baseline)", "RMSE"]
print(f"RMSE Holt-Winters (gedämpft) gegenüber Baseline: {verbesserung:.0%} kleiner")
metriken.round(2)

**Interpretation:**

| Modell | RMSE | MAE | MAPE |
|---|---|---|---|
| Holt-Winters (gedämpft) | 2,44 | 2,04 | 2,15 % |
| Seasonal Naive (Baseline) | 3,28 | 2,77 | 2,91 % |
| Holt-Winters (ungedämpft), nur zur Info | 4,69 | 3,97 | 4,20 % |

- Das gewählte Modell hat in allen drei Metriken kleinere Fehler als die Baseline; der RMSE ist 26 % kleiner.
- Mittlerer Fehler: gedämpft +0,81, Baseline −0,94 (Prognose im Mittel leicht zu hoch bzw. zu niedrig).
- Die ungedämpfte Variante ist schlechter als die Baseline. Ihr Trend am Trainingsende ist 0,296 pro Monat (gedämpft: 0,183) und wird ohne Dämpfung über alle 24 Monate fortgeschrieben; ihr mittlerer Fehler ist +3,67. Diese Zeile dient nur der Information: Gewählt wurde vorher per AIC.

## 5. Fehler je Testmonat

In [ ]:
fehler = hw.fehler_je_monat(test, {"Holt-Winters (gedämpft)": p_hw, "Seasonal Naive": p_sn})
fig = hw.plot_fehler(fehler)
speichern(fig, "07_hw_fehler")

e_hw, e_sn = fehler["Holt-Winters (gedämpft)"], fehler["Seasonal Naive"]
print(f"Monate mit kleinerem |Fehler| als Seasonal Naive: {(e_hw.abs() < e_sn.abs()).sum()} von {len(fehler)}")
print(f"Größter |Fehler| Holt-Winters: {e_hw.abs().max():.2f} ({e_hw.abs().idxmax():%m/%Y}, Fehler {e_hw[e_hw.abs().idxmax()]:+.2f})")
print(f"Größter |Fehler| Seasonal Naive: {e_sn.abs().max():.2f} ({e_sn.abs().idxmax():%m/%Y}, Fehler {e_sn[e_sn.abs().idxmax()]:+.2f})")
print(f"RMSE Holt-Winters: erste 12 Testmonate {evaluate(test[:12], p_hw[:12])['RMSE']:.2f}, letzte 12 {evaluate(test[12:], p_hw[12:])['RMSE']:.2f}")
fehler.round(2)

**Interpretation:**

- In 17 von 24 Testmonaten ist der absolute Fehler von Holt-Winters kleiner als der der Baseline.
- Größter Fehler Holt-Winters: +6,35 im 12/2024 (Prognose zu hoch). Auch 12/2025 wird überschätzt (+2,84).
- Größter Fehler Baseline: −7,10 im 04/2025 (Prognose zu niedrig).
- RMSE Holt-Winters im ersten Testjahr 2,69, im zweiten 2,15: Die Fehler werden im zweiten Prognosejahr nicht größer.

## Fazit und Einschränkungen

**Fazit:** Holt-Winters mit additiver Saison und gedämpftem Trend (Auswahl per AIC auf den Trainingsdaten) prognostiziert die 24 Testmonate mit RMSE 2,44, MAE 2,04 und MAPE 2,15 % und ist damit besser als Seasonal Naive (RMSE 3,28). Der Vergleich mit den ARIMA-Modellen folgt in `04_comparison.ipynb` mit denselben Metriken.

**Einschränkungen:**
1. Nur ein Testzeitraum (ein Prognoseursprung, 24 Monate). Das Ergebnis kann von diesem Zeitraum abhängen; eine Rolling-Origin-Evaluation wäre robuster.
2. $\beta$ liegt am Rand der Nebenbedingung $\beta \le \alpha$.
3. Es wurden nur zwei Varianten verglichen (Trend gedämpft/ungedämpft); die Saisonform (additiv) wurde aus der EDA übernommen.
4. Keine Prognoseintervalle für Holt-Winters berechnet.